<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Parking_Space.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
font_title = ImageFont.load_default()
font_stat = ImageFont.load_default()
font_badge = ImageFont.load_default()

In [10]:
import cv2
import pickle
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from google.colab.patches import cv2_imshow
from IPython.display import clear_output
from collections import defaultdict, deque
from google.colab import files

In [11]:
#Configuración

VIDEO_PATH = "/content/carPark.mp4"
POS_PATH = "/content/CarParkPos"

# Video de salida
OUTPUT_PATH = "/content/parking_result.mp4"

# Dimensiones de cada espacio
WIDTH = 107
HEIGHT = 48

In [12]:
# CARGAR POSICIONES DE LOS ESPACIOS

try:

    with open(POS_PATH, "rb") as f:
        posList = pickle.load(f)

    print(f"Espacios cargados: {len(posList)}")

except Exception as e:

    raise FileNotFoundError(
        f"No se pudo cargar {POS_PATH}.\n"
        f"Verifica que el archivo CarParkPos esté en /content/"
    )


if len(posList) == 0:
    raise ValueError(
        "CarParkPos está vacío. "
        "Necesitas definir las posiciones de los espacios."
    )



Espacios cargados: 69


In [24]:
print(posList)

[(52, 145), (52, 191), (53, 240), (50, 288), (51, 335), (52, 382), (52, 428), (53, 478), (55, 526), (54, 572), (56, 620), (51, 91), (158, 91), (158, 140), (157, 191), (158, 239), (157, 285), (157, 333), (159, 381), (160, 429), (161, 477), (160, 524), (162, 570), (163, 620), (507, 91), (399, 92), (399, 140), (400, 189), (401, 236), (400, 284), (401, 331), (402, 380), (402, 425), (403, 521), (406, 616), (506, 138), (508, 187), (509, 233), (508, 282), (509, 332), (510, 377), (510, 424), (512, 521), (511, 568), (513, 616), (748, 85), (748, 135), (748, 185), (748, 233), (749, 278), (750, 328), (751, 375), (749, 423), (751, 518), (752, 566), (752, 613), (901, 138), (904, 187), (906, 282), (905, 232), (907, 327), (908, 376), (909, 423), (910, 472), (911, 519), (912, 567), (911, 612), (750, 475), (400, 571)]


In [13]:
# ABRIR VIDEO

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(
        f"No se pudo abrir el video: {VIDEO_PATH}"
    )


fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 30


frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Resolución:", frame_width, "x", frame_height)
print("FPS:", fps)
print("Frames:", total_frames)


Resolución: 1100 x 720
FPS: 24.0
Frames: 679


In [14]:
# VIDEO DE SALIDA

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (frame_width, frame_height)
)


In [15]:
def process_frame(img):
    imgGray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    imgBlur = cv2.GaussianBlur(imgGray, (3, 3), 1)
    imgThreshold = cv2.adaptiveThreshold(imgBlur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                         cv2.THRESH_BINARY_INV, 25, 16)
    imgMedian = cv2.medianBlur(imgThreshold, 5)
    kernel = np.ones((3, 3), np.uint8)
    return cv2.dilate(imgMedian, kernel, iterations=1)

In [18]:
def draw_modern_ui(img, imgDilate):
    overlay = img.copy()
    free_count = 0
    occupied_count = 0

    for pos in posList:
        x, y = pos
        imgCrop = imgDilate[y:y+height, x:x+width]
        count = cv2.countNonZero(imgCrop)

        if count < 800:
            free_count += 1
            cv2.rectangle(overlay, (x, y), (x + width, y + height), (46, 204, 113), -1)
            cv2.rectangle(img, (x, y), (x + width, y + height), (46, 204, 113), 2)
        else:
            occupied_count += 1
            cv2.rectangle(overlay, (x, y), (x + width, y + height), (60, 64, 235), -1)
            cv2.rectangle(img, (x, y), (x + width, y + height), (60, 64, 235), 2)

    alpha = 0.28
    img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil, 'RGBA')

    hud_x = 20
    hud_y = 10
    hud_w = 720
    hud_h = 58

    draw.rounded_rectangle([hud_x, hud_y, hud_x + hud_w, hud_y + hud_h],
                            radius=14, fill=(15, 23, 42, 240), outline=(255, 255, 255, 60), width=1)

    draw.text((hud_x + 16, hud_y + 11), 'ESTACIONAMIENTO CON IA', font=font_title, fill=(255, 255, 255, 255))
    draw.ellipse([hud_x + 16, hud_y + 36, hud_x + 24, hud_y + 44], fill=(46, 204, 113, 255))
    draw.text((hud_x + 28, hud_y + 33), 'MONITOREO EN TIEMPO REAL', font=font_badge, fill=(46, 204, 113, 255))

    total_spaces = len(posList)
    occupancy_rate = int((occupied_count / total_spaces) * 100) if total_spaces > 0 else 0

    draw.ellipse([hud_x + 230, hud_y + 16, hud_x + 242, hud_y + 28], fill=(46, 204, 113, 255))
    draw.text((hud_x + 248, hud_y + 13), f'LIBRES: {free_count}/{total_spaces}', font=font_stat, fill=(240, 240, 240, 255))

    draw.ellipse([hud_x + 365, hud_y + 16, hud_x + 377, hud_y + 28], fill=(235, 64, 60, 255))
    draw.text((hud_x + 383, hud_y + 13), f'OCUPADOS: {occupied_count}/{total_spaces}', font=font_stat, fill=(240, 240, 240, 255))

    draw.ellipse([hud_x + 535, hud_y + 16, hud_x + 547, hud_y + 28], fill=(59, 130, 246, 255))
    draw.text((hud_x + 553, hud_y + 13), f'TASA: {occupancy_rate}%', font=font_stat, fill=(240, 240, 240, 255))

    pb_x, pb_y, pb_w, pb_h = hud_x + 230, hud_y + 38, 465, 8
    draw.rounded_rectangle([pb_x, pb_y, pb_x + pb_w, pb_y + pb_h], radius=4, fill=(40, 50, 65, 255))
    fill_w = int(pb_w * (occupied_count / total_spaces)) if total_spaces > 0 else 0
    if fill_w > 0:
        draw.rounded_rectangle([pb_x, pb_y, pb_x + fill_w, pb_y + pb_h], radius=4, fill=(235, 64, 60, 255))

    for pos in posList:
        x, y = pos
        imgCrop = imgDilate[y:y+height, x:x+width]
        count = cv2.countNonZero(imgCrop)
        is_free = count < 800

        bg_color = (46, 204, 113, 230) if is_free else (235, 64, 60, 230)
        text_str = 'LIBRE' if is_free else 'OCUPADO'

        bw, bh = 54, 16
        bx, by = x + width - bw - 4, y + height - bh - 4
        draw.rounded_rectangle([bx, by, bx + bw, by + bh], radius=4, fill=bg_color)

        tx = bx + 11 if is_free else bx + 3
        draw.text((tx, by + 1), text_str, font=font_badge, fill=(255, 255, 255, 255))

        img = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

    return img

In [21]:
# PROCESAMIENTO

frame_count = 0

while True:
  success, img = cap.read()

  if not success:
    break

  imgDilate = process_frame(img)
  img = draw_modern_ui(img, imgDilate)

  frame_count += 1

  out.write(img)

  clear_output(wait=True)

cap.release()
out.release()

In [22]:
files.download(OUTPUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>